# AI Wine Sommelier RAG

## Wine Review Indexing

https://www.kaggle.com/datasets/christopheiv/winemagdata130k

In [3]:
%pip install -Uqqq langchain langchain-community langchain-openai langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [9]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## Pincone 테스트

In [10]:
# 데이터로드
from langchain_core.documents import Document

documents = [
    Document(page_content="LangChain은 LLM 기반 애플리케이션을 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://langchain.com/docs", "author": "alice", "page": 1}),
    Document(page_content="ChromaDB는 오픈소스 벡터 데이터베이스입니다.", metadata={"source": "https://chromadb.org/intro", "license": "MIT", "date": "2024-07-01"}),
    Document(page_content="파이썬으로 AI 서비스를 개발할 수 있습니다.", metadata={"source": "https://pythonai.co.kr", "editor": "kim", "page": 7}),
    Document(page_content="LLM은 자연어 처리를 위한 대형 언어 모델을 의미합니다.", metadata={"source": "https://llmwiki.com/info", "author": "bob", "version": "v1.1"}),
    Document(page_content="RAG는 검색과 생성의 결합 방식을 제공합니다.", metadata={"source": "https://rag-search.io", "reviewer": "lee", "section": "summary"}),
    Document(page_content="벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.", metadata={"source": "https://vectorbase.net", "author": "jin", "topic": "vector"}),
    Document(page_content="LangChain을 이용하면 다양한 AI 파이프라인을 구축할 수 있습니다.", metadata={"source": "https://langchain.com/blog", "editor": "sarah", "date": "2024-06-30"}),
    Document(page_content="OpenAI의 GPT 모델은 텍스트 생성에 특화되어 있습니다.", metadata={"source": "https://openai.com/gpt", "lang": "ko", "page": 5}),
    Document(page_content="파이썬은 AI 및 데이터 분석 분야에서 널리 사용되는 언어입니다.", metadata={"source": "https://python.org/usecases", "author": "chun", "updated": "2024-05"}),
    Document(page_content="Streamlit은 파이썬으로 대시보드를 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://streamlit.io/start", "editor": "park", "date": "2024-04-28"}),
    Document(page_content="Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.", metadata={"source": "https://retrieval.ai/dense", "type": "tech", "page": 3}),
    Document(page_content="Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.", metadata={"source": "https://pandas.pydata.org/about", "maintainer": "koh", "section": "intro"}),
    Document(page_content="메타데이터 필터링은 검색 결과의 품질을 높여줍니다.", metadata={"source": "https://search.com/metadata", "author": "seo", "feature": "filter"}),
    Document(page_content="SelfQueryRetriever는 자연어 쿼리를 임베딩 쿼리로 변환해줍니다.", metadata={"source": "https://selfquery.ai", "editor": "min", "date": "2024-05-12"}),
    Document(page_content="프롬프트 엔지니어링은 LLM의 성능을 극대화하는 방법입니다.", metadata={"source": "https://prompting.dev/guide", "author": "yang", "topic": "prompt"}),
    Document(page_content="HyDE 기법은 하이브리드 검색에 사용됩니다.", metadata={"source": "https://hyde-tech.com", "reviewer": "kang", "version": "2024.1"}),
    Document(page_content="CoT는 복잡한 문제를 단계적으로 해결하는 프롬프트 기법입니다.", metadata={"source": "https://cotprompt.org", "editor": "jung", "date": "2023-12-01"}),
    Document(page_content="문서 임베딩은 텍스트를 고차원 벡터로 변환하는 과정입니다.", metadata={"source": "https://embedding.ai/intro", "section": "embedding", "author": "song"}),
    Document(page_content="CrewAI는 멀티 에이전트 시스템 구현을 돕는 툴입니다.", metadata={"source": "https://crew.ai/docs", "lang": "ko", "page": 9}),
    Document(page_content="Fine-tuning은 사전학습 모델을 특정 도메인에 맞게 재학습시키는 과정입니다.", metadata={"source": "https://finetune.ai/guide", "editor": "jeon", "date": "2024-01-30"})
]

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1536차원 임베딩 모델
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small') 

# 문서 -> 임베딩 -> Pinecorn 업로드
vector_store = PineconeVectorStore.from_documents(
    documents, # list[Document]
    embeddings,
    index_name = 'pincone-test' # 저장할 index명
)

In [15]:
# 코사인 유사도로 검색
retriever = vector_store.similarity_search('벡터 데이터베이스란?')
retriever

[Document(id='d796f828-fce5-4005-8ec3-2529569ca766', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='de804a6d-dae2-4224-91a7-809673e6524f', metadata={'date': '2024-07-01', 'license': 'MIT', 'source': 'https://chromadb.org/intro'}, page_content='ChromaDB는 오픈소스 벡터 데이터베이스입니다.'),
 Document(id='226a422d-6f78-4867-8ea9-3866dc4eaec7', metadata={'page': 3.0, 'source': 'https://retrieval.ai/dense', 'type': 'tech'}, page_content='Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.'),
 Document(id='50453ced-1611-4369-b4c1-62d2a0121387', metadata={'maintainer': 'koh', 'section': 'intro', 'source': 'https://pandas.pydata.org/about'}, page_content='Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.')]

In [ ]:
# 벡터스토어를 Retriever 인터페이스로 변환
retriever = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 5}
)

retriever.invoke('벡터 데이터베이스란?')

[Document(id='d796f828-fce5-4005-8ec3-2529569ca766', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='de804a6d-dae2-4224-91a7-809673e6524f', metadata={'date': '2024-07-01', 'license': 'MIT', 'source': 'https://chromadb.org/intro'}, page_content='ChromaDB는 오픈소스 벡터 데이터베이스입니다.'),
 Document(id='226a422d-6f78-4867-8ea9-3866dc4eaec7', metadata={'page': 3.0, 'source': 'https://retrieval.ai/dense', 'type': 'tech'}, page_content='Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.'),
 Document(id='50453ced-1611-4369-b4c1-62d2a0121387', metadata={'maintainer': 'koh', 'section': 'intro', 'source': 'https://pandas.pydata.org/about'}, page_content='Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.'),
 Document(id='09a19221-6a64-45fb-a49b-644274dc9702', metadata={'author': 'song', 'section': 'embedding', 'source': 'https://embedding.ai/intro'}, page_content='문서 임베딩은 텍스트를 고차원 벡터로 변환하는 과정입니다.')]

In [17]:
!gdown 1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3

Downloading...
From: https://drive.google.com/uc?id=1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3
To: c:\Users\playdata2\LLM\06_2stage_rag\winemag-data-130k-v2.csv

  0%|          | 0.00/52.9M [00:00<?, ?B/s]
  1%|          | 524k/52.9M [00:00<00:25, 2.08MB/s]
  3%|▎         | 1.57M/52.9M [00:00<00:10, 4.92MB/s]
  9%|▉         | 4.72M/52.9M [00:00<00:03, 13.3MB/s]
 20%|█▉        | 10.5M/52.9M [00:00<00:01, 26.2MB/s]
 28%|██▊       | 14.7M/52.9M [00:00<00:01, 25.2MB/s]
 44%|████▎     | 23.1M/52.9M [00:00<00:00, 39.8MB/s]
 54%|█████▎    | 28.3M/52.9M [00:00<00:00, 42.3MB/s]
 63%|██████▎   | 33.6M/52.9M [00:01<00:00, 44.0MB/s]
 72%|███████▏  | 38.3M/52.9M [00:01<00:00, 44.6MB/s]
 81%|████████▏ | 43.0M/52.9M [00:01<00:00, 41.7MB/s]
 92%|█████████▏| 48.8M/52.9M [00:01<00:00, 44.6MB/s]
100%|██████████| 52.9M/52.9M [00:01<00:00, 34.0MB/s]


In [21]:
# CSVLoader : CSV의 각 행을 하나의 Document로 변환하는 로더
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader('winemag-data-130k-v2.csv', encoding='utf-8')
docs = loader.load() # CSV를 -> list[Document]
print(len(docs))

129971


In [22]:
for i, doc in enumerate(docs[:2]): # 상위 2개만 확인
    print(f"{i}: type(doc)")
    print(f"{doc.metadata}")
    print(f"{doc.page_content}")
    print()

0: type(doc)
{'source': 'winemag-data-130k-v2.csv', 'row': 0}
: 0
country: Italy
description: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
designation: Vulkà Bianco
points: 87
price: 
province: Sicily & Sardinia
region_1: Etna
region_2: 
taster_name: Kerin O’Keefe
taster_twitter_handle: @kerinokeefe
title: Nicosia 2013 Vulkà Bianco  (Etna)
variety: White Blend
winery: Nicosia

1: type(doc)
{'source': 'winemag-data-130k-v2.csv', 'row': 1}
: 1
country: Portugal
description: This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red berry fruits and freshened with acidity. It's  already drinkable, although it will certainly be better from 2016.
designation: Avidagos
points: 87
price: 15.0
province: Douro
region_1: 
region_2: 
taster_name: Roger Voss
taster_twitter_handle: @vossroger
title: Quinta dos Avidagos

In [24]:
# Pinecone 업로드 : Pinecone 인덱스에 연결된 벡터스토어 객체 생성
vector_store = PineconeVectorStore(
    index_name = 'winemag-data',
    embedding= embeddings
)

batch_size = 100

for i in range(0, len(docs), batch_size): # 전체 docs를 batch_size 단위로 순회
    batch_data = docs[i: i + batch_size]                         # 해당 인덱스 + 100 씩 리스트 가져옴
    vector_store.add_documents(batch_data)            # 배치 Document들을 임베딩 -> Pinecone 업로드
    print(f"index: {i} ~ {i + batch_size}")


index: 0 ~ 100
index: 100 ~ 200
index: 200 ~ 300
index: 300 ~ 400
index: 400 ~ 500
index: 500 ~ 600
index: 600 ~ 700
index: 700 ~ 800
index: 800 ~ 900
index: 900 ~ 1000
index: 1000 ~ 1100
index: 1100 ~ 1200
index: 1200 ~ 1300
index: 1300 ~ 1400
index: 1400 ~ 1500
index: 1500 ~ 1600
index: 1600 ~ 1700
index: 1700 ~ 1800
index: 1800 ~ 1900
index: 1900 ~ 2000
index: 2000 ~ 2100
index: 2100 ~ 2200
index: 2200 ~ 2300
index: 2300 ~ 2400
index: 2400 ~ 2500
index: 2500 ~ 2600
index: 2600 ~ 2700
index: 2700 ~ 2800
index: 2800 ~ 2900
index: 2900 ~ 3000
index: 3000 ~ 3100
index: 3100 ~ 3200
index: 3200 ~ 3300
index: 3300 ~ 3400
index: 3400 ~ 3500
index: 3500 ~ 3600
index: 3600 ~ 3700
index: 3700 ~ 3800
index: 3800 ~ 3900
index: 3900 ~ 4000
index: 4000 ~ 4100
index: 4100 ~ 4200
index: 4200 ~ 4300
index: 4300 ~ 4400
index: 4400 ~ 4500
index: 4500 ~ 4600
index: 4600 ~ 4700
index: 4700 ~ 4800
index: 4800 ~ 4900
index: 4900 ~ 5000
index: 5000 ~ 5100
index: 5100 ~ 5200
index: 5200 ~ 5300
index: 5300 ~ 

KeyboardInterrupt: 

## Retrieval & Generation
1. 텍스트/이미지 입력으로 요리에 설명 chain
2. 요리설명텍스트 벡터db조회 chain
3. 요리설명/리뷰검색을 가지고 와인추천 응답 chain

### 요리설명 chain

In [ ]:
# 채팅 프롬프트 / 휴먼 메시지 템플릿
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser # 출력 -> 문자열 파싱

# 함수를 Runnable로 감싸 chain에서 실행
from langchain_core.runnables import RunnableLambda

def describe_dish_flavor(query:dict):
    prompt = ChatPromptTemplate.from_messages([
        ('system', ''' 
**페르소나 (Persona):**
당신은 식재료의 분자 단위까지 이해하는 '미식의 철학자'이자, 절대미각을 지닌 최고 수준의 푸드 칼럼니스트이다.
당신은 요리를 단순한 음식이 아닌, 식재료와 조리 과학(Culinary Science)이 빚어낸 예술 작품으로 바라본다.
당신의 표현은 식재료의 기원부터 조리 과정에서 일어나는 화학적 변화(마이야르 반응, 캐러멜라이징 등)를 아우르며, 읽는 이가 마치 그 음식을 입안에 넣은 듯한 착각을 불러일으킬 정도로 정교하고 관능적이다.

**역할 (Role):**
당신의 핵심 역할은 요리의 맛, 향, 텍스처(Texture), 그리고 밸런스를 해부학적으로 분석하여 전달하는 것이다.
1.  **다차원적 분석:** 맛을 평면적으로 묘사하지 않고, '첫맛(Attack) - 중간 맛(Mid-palate) - 끝맛(Finish)'의 시퀀스로 나누어 입체적으로 설명한다.
2.  **조리법과 맛의 인과관계:** 왜 이 맛이 나는지, 어떤 조리 테크닉이 식재료의 잠재력을 폭발시켰는지 논리적 근거를 제시한다.
3.  **미식의 가이드:** 식재료 간의 궁합(Pairing)과 풍미를 극대화하는 팁을 제공하여, 사용자의 미식 수준을 한 단계 끌어올린다.

**가이드라인 (Guidelines):**
- **감각의 구체화:** '맛있다', '부드럽다' 같은 추상적 표현을 금지한다. 대신 '혀를 감싸는 벨벳 같은 질감', '비강을 때리는 훈연 향' 등 구체적인 묘사를 사용하라.
- **단계별 서술:** 시각과 후각으로 시작해, 입안에서의 질감 변화, 그리고 목 넘김 후의 여운까지 단계별로 서술하라.

**예시 (Examples):**

* **사용자:** "잘 만든 '트러플 크림 리조또'의 맛을 묘사해 주세요."
    **당신:**
    * **[시각과 후각]** 김이 모락모락 나는 접시 위로 흙내음(Earthy)을 가득 머금은 트러플 향이 가장 먼저 코끝을 강타합니다. 크림소스의 녹진한 유분 향과 섞여 마치 가을 숲속에 와 있는 듯한 묵직한 아로마가 식욕을 자극합니다.
    * **[첫맛과 텍스처]** 한 숟가락 입에 넣으면, 알덴테(Al dente)로 익혀 심지가 살아있는 쌀알이 혀 위에서 경쾌하게 굴러다닙니다. 동시에 파르미지아노 레지아노 치즈가 녹아든 크림소스가 쌀알 사이사이를 끈적하게 메우며 혀를 포근하게 감싸 안습니다.
    * **[풍미의 폭발]** 씹을수록 버섯의 감칠맛(Umami)이 폭발합니다. 버터의 고소함이 베이스를 깔아주는 가운데, 트러플 오일의 강렬한 향이 비강으로 역류하며 미각을 지배합니다.
    * **[여운]** 목을 넘긴 후에도 트러플의 진한 향과 크림의 고소함이 입안에 길게 남아, 무거운 레드 와인 한 모금을 간절하게 부릅니다.

* **사용자:** "양파 수프(French Onion Soup)의 맛의 비결이 무엇인가요?"
    **당신:**
    * **[핵심 분석]** 이 요리의 영혼은 **'인내심이 만든 단맛'**에 있습니다. 양파를 약불에서 장시간 볶아내는 '캐러멜라이징(Caramelization)' 과정이 핵심입니다.
    * **[맛의 레이어]** 양파의 매운 성분이 열을 만나 짙은 갈색의 끈적한 당분으로 변하며, 설탕과는 차원이 다른 깊고 복합적인 단맛을 냅니다. 여기에 쇠고기 육수의 짭조름한 감칠맛이 더해져 '단짠'의 완벽한 균형을 이룹니다.
    * **[식감의 조화]** 흐물흐물하게 녹아내린 양파와 국물을 머금어 축축해진 바게트, 그리고 그 위를 덮은 그뤼에르 치즈의 쫄깃함이 섞이며 입안 가득 풍성한 식감의 축제를 엽니다.

**주의사항**
맛의 대한 묘사만 줄글 형식으로 50자이내로 작성하세요.
'''),
('human', '사용자가 제공한 이미지의 요리명과 풍미를 잘 묘사해 주세요.')
    ])

    temp = []
    # image_urls 가 있는 경우 이미지 URL들을 메시지 블록으로 추가
    if query.get('image_urls'):
        temp += [{"image_url" : image_url} for image_url in query.get('image_urls')]
    # text 가 있는 경우 메시지 블록으로 추가
    if query.get('text'):
        temp+=[{"text": query.get('text')}]

    # HumanMessagePromptTemplate : 멀티모달 블록형태의 값을 human 메시지로 프롬프트에 추가    
    prompt += HumanMessagePromptTemplate.from_template(temp)

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain # 체인 객체 반환

# 입력을 받아 chain을 실행할 수 있는 Runnable
dish_flavor_chain = RunnableLambda(describe_dish_flavor)
response = dish_flavor_chain.invoke({
    "text" : "",
    "image_urls": [
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMjVfODIg%2FMDAxNzYxMzc2NjMzMDA4.-qxYpSDZfPleD8cj9VzxvqckYRIvaGpZW-fibT3whjsg.mefkB_k7NzVsXrb9RGPEQvZAplyzrustInLMV-827Gkg.JPEG%2FIMG%25A3%25DF2865.JPG&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTA5MjBfMjEz%2FMDAxNzU4MzgwMDM4MzAz.XWaCn8Xu_7jjbYWq5P4MFqibaqNz4p3CFKRgjOnP2dMg.7wrfjF9U-p-CORf9ix4DbEGFRnOkaNh2ihjYlZOZy6Ag.JPEG%2FIMG_6083.JPG&type=sc960_832"
    ]
})

print(response)

바삭한 돈가스와 매콤달콤한 제육볶음, 고소한 풍미가 어우러진 한상입니다.


## 리뷰 검색 chain

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 요리 풍미 설명을 받아서 Pinecone에서 유사한 와인리뷰를 찾아 반환하는함수
def search_wine_review(query):
    # 1536차원 임베딩 모델
    embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small') 

    # PineconeVetorStore index 연결
    PineconeVectorStore(
        index_name="winemag-data", # 검색할 index 명
        embedding=embeddings, # 질의문 임베딩에 사용할 모델
    )

    docs = vector_store.similarity_search(query, k=5)

    return {
        'dish_flavor' : query,
        'wine_reviews' : '\n\n'.join(doc.page_content for doc in docs)
    }

query = ''' 
첫 번째 이미지는 '허브 그릴 스테이크'입니다. 입안에서 육즙과 허브 오일이 조화를 이루며 풍부하고 깊은 감칠맛이 혀를 감싸고, 구운 토마토의 은은한 산미가 중간 맛에 신선함을 부여합니다.
두 번째 이미지는 '시저 샐러드'입니다. 크리스피한 크루통과 신선한 로메인 상추가 바삭한 질감을 선사하고, 고소한 파마산 치즈와 크리미한 시저 드레싱이 입안 가득 고소함과 산뜻한 여운을 남깁니다.
'''

result = search_wine_review(query) # {'dish_flavor' : ..., 'wine_reviews' : ...}
print(result['wine_reviews'])

: 12195
country: US
description: Soft and melted in texture, with flavors of cherry jam, cassis, chocolate mints, anise and pepper. Simple, but tasty.
designation: Sophie's Romp
points: 84
price: 20.0
province: California
region_1: Napa County
region_2: Napa
taster_name: 
taster_twitter_handle: 
title: Punk Dog 2004 Sophie's Romp Red (Napa County)
variety: Red Blend
winery: Punk Dog

: 8158
country: US
description: A sweet smell of honeysuckle and grapefruit candy permeates the nose of this bottling, along with cut honeydew melon and apple blossom. Sugary mandarin juice is the primary flavor on the palate, but it's properly offset by acidity a chalky texture.
designation: 
points: 87
price: 29.0
province: California
region_1: Central Coast
region_2: 
taster_name: Matt Kettmann
taster_twitter_handle: @mattkettmann
title: Coquelicot 2016 Riesling
variety: Riesling
winery: Coquelicot

: 4988
country: US
description: Honey-sweet and direct, with citrus jam, apricot essence, crème brûlée an

In [ ]:
# 파이프라인 중간점검 (요리 풍미 추출 -> 와인 리뷰 검색)

search_wine_review_chain = RunnableLambda(search_wine_review) # 함수 -> Runnable

# 이미지, 텍스트 -> 풍미 | 풍미 -> 리뷰 검색
chain = dish_flavor_chain | search_wine_review_chain

response = chain.invoke({
    "text": "",
    "image_urls" :[
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%252866%2529.png&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832"
    ]
})

print(response)

{'dish_flavor': '허브 스테이크는 육즙과 불향이 진하고, 시저 샐러드는 고소·상큼하다.', 'wine_reviews': ": 26833\ncountry: US\ndescription: The trick with sparkling wine is to achieve finesse. This Pinot Noir-Chardonnay blend is too scoury in bubbles, giving it a rough feel. Nonetheless it's delicious and easy to like for its yeasty flavors of limes, oranges and vanilla honey.\ndesignation: Brut\npoints: 87\nprice: 45.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: \ntaster_twitter_handle: \ntitle: Kessler-Haak 2012 Brut Sparkling (Sta. Rita Hills)\nvariety: Sparkling Blend\nwinery: Kessler-Haak\n\n: 29012\ncountry: US\ndescription: Shows many of the qualities of Schramsberg's more expensive sparklers, except the bubbles aren't quite as refined. The flavors are rich and satisfying in strawberries, raspberries, vanilla and toast.\ndesignation: Mirabelle Brut\npoints: 87\nprice: 23.0\nprovince: California\nregion_1: North Coast\nregion_2: North Coast\ntaster_name: \ntaster_tw

In [33]:
response = chain.invoke({"text": "오늘 저녁은 버터와 허브로 구운 캐비어, 가리비 관자 구이를 먹겠다."})

print(response)

{'dish_flavor': '버터와 허브 향을 머금은 캐비어·가리비 관자 구이, 짭조름한 감칠맛과 달큰한 육즙이 어우러진다.', 'wine_reviews': ": 15950\ncountry: Spain\ndescription: Inky, minerally aromas of blackberry, black plum and coconut filter into a round, fluffy palate that's friendly and pure but not very dense or structured. Baked flavors of molasses and gamy berry finish mild and easy.\ndesignation: Viñas de Gain\npoints: 90\nprice: 25.0\nprovince: Northern Spain\nregion_1: Rioja\nregion_2: \ntaster_name: Michael Schachner\ntaster_twitter_handle: @wineschach\ntitle: Artadi 2011 Viñas de Gain  (Rioja)\nvariety: Tempranillo\nwinery: Artadi\n\n: 3953\ncountry: Spain\ndescription: Starts out with leather and cheesy aromas, and beyond that there's not a lot of fruit on the bouquet. It's a chunky, heavy wine with thick, jammy plum flavors that have strong herbal undertones. Warm, grabby and baked on the finish, which is plodding.\ndesignation: Tinto\npoints: 83\nprice: 10.0\nprovince: Northern Spain\nregion_1: Calatayud\nregion_2: \n

In [36]:
# 와인 추천 체인 : 요리 풍미 + 검색된 와인 리뷰를 바탕으로 페어링 추천
def recommend_wines(query):
    prompt = ChatPromptTemplate.from_messages([
        ('system', ''' 
    **페르소나 (Persona):**
당신은 와인과 미식의 조화로운 세계를 탐험하는 '마리아주(Mariage)의 설계자'이자 경험 풍부한 소믈리에이다.
당신은 전 세계의 와인 산지와 품종에 대한 백과사전적 지식을 갖추고 있으며, 복잡한 와인 용어를 누구나 이해하기 쉬운 감각적인 언어로 풀어내는 탁월한 능력을 지녔다.
당신의 태도는 언제나 환대하는 마음(Hospitality)으로 가득 차 있어, 와인 초보자부터 애호가까지 모두를 편안하게 이끈다.

**역할 (Role):**
당신의 유일하고도 가장 중요한 역할은 사용자가 준비한 요리에 **'영혼의 단짝'이 될 와인을 추천**하는 것이다.
1.  **미각 분석:** 요리의 주재료, 소스, 조리법(굽기, 찌기 등)을 분석하여 맛의 무게감과 특성을 파악한다.
2.  **정밀한 페어링:** 산도(Acidity), 당도(Sweetness), 타닌(Tannin), 바디감(Body)의 균형을 고려해 와인을 선정한다.
3.  **이유 설명:** 단순히 와인 이름만 던지는 것이 아니라, **"왜 이 와인이 그 음식과 어울리는지"** 미각적, 화학적 근거를 들어 설득력 있게 설명한다.

**가이드라인 (Guidelines):**
- **음식 중심 예시:** 모든 답변은 구체적인 요리에 대한 와인 추천으로 이루어져야 한다.
- **상호보완의 원리:** 와인이 음식의 맛을 어떻게 상승시키는지(증폭), 혹은 음식의 단점을 어떻게 가려주는지(보완) 묘사하라.

**예시 (Examples):**
... (생략) ...
'''),
                # 입력 변수(dish_flavor, wine_reviews) 기반 요청
        ('human', '''
와인페이링 추천에 있어 아래 제시된 요리와 풍미, 와인리뷰만을 기초하여 답변해주세요.

## 요리와 풍미 ##
{dish_flavor}

## 와인리뷰 정보 ##
{wine_reviews}
''')
    ])

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain # 체인 결과 반환

recommend_wines_chain = RunnableLambda(recommend_wines)
response = recommend_wines_chain.invoke({
    'dish_flavor': '버터와 허브 향을 머금은 캐비어·가리비 관자 구이, 짭조름한 감칠맛과 달큰한 육즙이 어우러진다.',
    'wine_reviews': ": 15950\ncountry: Spain\ndescription: Inky, minerally aromas of blackberry, black plum and coconut filter into a round, fluffy palate that's friendly and pure but not very dense or structured. Baked flavors of molasses and gamy berry finish mild and easy.\ndesignation: Viñas de Gain\npoints: 90\nprice: 25.0\nprovince: Northern Spain\nregion_1: Rioja\nregion_2: \ntaster_name: Michael Schachner\ntaster_twitter_handle: @wineschach\ntitle: Artadi 2011 Viñas de Gain  (Rioja)\nvariety: Tempranillo\nwinery: Artadi\n\n: 3953\ncountry: Spain\ndescription: Starts out with leather and cheesy aromas, and beyond that there's not a lot of fruit on the bouquet. It's a chunky, heavy wine with thick, jammy plum flavors that have strong herbal undertones. Warm, grabby and baked on the finish, which is plodding.\ndesignation: Tinto\npoints: 83\nprice: 10.0\nprovince: Northern Spain\nregion_1: Calatayud\nregion_2: \ntaster_name: Michael Schachner\ntaster_twitter_handle: @wineschach\ntitle: Figaro 2009 Tinto Red (Calatayud)\nvariety: Red Blend\nwinery: Figaro\n\n: 3895\ncountry: Spain\ndescription: Darker and earthier than most basic Garnachas, with aromas of compost, baked earth, crusty black fruits and leather. The palate has a nice, ripe, bodied feel, while flavors of blackberry and boysenberry bring just the slightest hint of herbal green. Toasty and solid on the finish.\ndesignation: Kickass\npoints: 86\nprice: 13.0\nprovince: Northern Spain\nregion_1: Cariñena\nregion_2: \ntaster_name: Michael Schachner\ntaster_twitter_handle: @wineschach\ntitle: El Burro 2008 Kickass Garnacha (Cariñena)\nvariety: Garnacha\nwinery: El Burro\n\n: 34807\ncountry: Spain\ndescription: Thick and pasty smelling, this has roasted berry scents that are slightly leathery. It's scratchy and mildly astringent on the palate, tasting minty and dry, with a mild fruit content. The finish is chunky and resiny.\ndesignation: Ribota\npoints: 84\nprice: 13.0\nprovince: Central Spain\nregion_1: Vino de la Tierra de Castilla\nregion_2: \ntaster_name: Michael Schachner\ntaster_twitter_handle: @wineschach\ntitle: Mano A Mano 2009 Ribota Tempranillo (Vino de la Tierra de Castilla)\nvariety: Tempranillo\nwinery: Mano A Mano\n\n: 39934\ncountry: Spain\ndescription: Leathery and rubbery at first, with aromatic accents of celery seed, barnyard and melted tar. The palate is lighter than the nose, with herbal plum flavors and a lean, strained body. Thin and fresh, but nothing special.\ndesignation: \npoints: 84\nprice: 17.0\nprovince: Northern Spain\nregion_1: Ribera del Duero\nregion_2: \ntaster_name: Michael Schachner\ntaster_twitter_handle: @wineschach\ntitle: De Lozar 2007  Ribera del Duero\nvariety: Tempranillo Blend\nwinery: De Lozar"
})

print(response)

### 추천 와인: **Artadi 2011 Viñas de Gain Rioja — Tempranillo**

이 요리에는 제시된 와인 중 **Artadi Viñas de Gain**이 가장 잘 어울립니다.

- **버터와 가리비의 달큰한 육즙**: 와인의 둥글고 부드러운 질감과 블랙베리·블랙플럼 풍미가 가리비의 은은한 단맛을 받쳐줍니다.
- **허브 향**: 와인에 느껴지는 감미로운 베리와 약간의 감칠맛·가메이한 뉘앙스가 허브 풍미와 자연스럽게 연결됩니다.
- **캐비어의 짠맛과 감칠맛**: 와인이 “친화적이고 순수하며”, 구조감이 지나치게 강하지 않고 마무리도 온화해 캐비어의 섬세한 풍미를 압도할 가능성이 상대적으로 낮습니다.
- **버터의 풍부함**: 와인의 둥근 팔레트가 버터의 농도를 받아주면서, 미네랄 느낌이 음식의 짭조름한 인상을 정리해줄 수 있습니다.

### 다른 후보보다 나은 이유

- **Figaro Tinto**와 **El Burro Garnacha**는 무겁고 잼 같은 과실감, 허브·구운 뉘앙스가 강해 가리비와 캐비어의 섬세함을 덮을 수 있습니다.
- **Mano a Mano Tempranillo**는 거칠고 약간 떫은 질감이 있어 캐비어의 염도와 충돌할 가능성이 있습니다.
- **De Lozar Ribera del Duero**는 향에서 가죽·타르·바닐라 계열의 거친 인상이 두드러져, 버터와 허브를 곁들인 관자의 섬세한 단맛과는 거리가 있습니다.

**결론적으로, 이 요리에는 과도하게 무겁거나 떫지 않고 둥글고 부드러운 Artadi Viñas de Gain이 가장 적합합니다.**


## 통합 Chain

In [ ]:
# {'text' : ..., 'image_urls': ... } -> 요리 풍미(텍스트)
dish_flavor_chain = RunnableLambda(describe_dish_flavor) 
# 풍미 텍스트 -> 유사한 와인 리뷰 검색 -> {'dish_flavor': ..., 'wine_reviews':...}
search_wine_review_chain = RunnableLambda(search_wine_review)
# {'dish_flavor': ..., 'wine_reviews':...} -> 최종 와인 페어링 추천
recommend_wines_chain = RunnableLambda(recommend_wines)

chain = dish_flavor_chain | search_wine_review_chain | recommend_wines_chain

response = chain.invoke({
    'text': "",
    "image_urls":[
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%252866%2529.png&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832"
    ]
})

print(response)

## 최종 추천  
### **Schramsberg NV Mirabelle Brut Sparkling**
**87점 · $23 · 미국 캘리포니아**

허브 시어링 스테이크와 시저 샐러드를 함께 즐긴다면 이 와인이 가장 균형이 좋습니다.

- **스테이크와의 조화:** 리뷰에 나타난 **딸기·라즈베리의 풍성한 과실미**가 육즙의 고소함을 부드럽게 받쳐줍니다. 여기에 **바닐라와 토스트 풍미**가 허브 시어링의 구운 향, 시저 샐러드의 바삭한 크루통과 자연스럽게 연결됩니다.
- **샐러드와의 조화:** 스파클링의 기포가 스테이크의 기름진 질감을 씻어내 입안을 새롭게 해주고, 샐러드의 상큼한 소스와도 대비를 만들어 줍니다.
- **전체적인 인상:** 리뷰에서 “풍부하고 만족스럽다”고 평가된 스타일이라, 가벼운 샐러드뿐 아니라 육즙 있는 스테이크까지 감당할 수 있습니다.

다만 리뷰상 기포가 아주 세련되지는 않다고 되어 있으므로, 섬세하고 날카로운 느낌보다는 **풍성하고 편안한 식사형 페어링**으로 접근하는 것이 좋습니다.

## 샐러드 중심이라면  
### **Freixenet NV Elyssia Gran Cuvée Brut Cava**
**88점 · $18**

레몬, 시트러스, 천도복숭아, 오렌지의 **신선하고 중간 정도의 바디감**이 시저 소스의 상큼함을 가장 잘 살려줍니다. 크루통의 고소함도 산뜻하게 정리해주지만, 스테이크의 육즙과 고소한 풍미에는 Schramsberg보다 다소 가볍게 느껴질 수 있습니다.

## 더 날카롭고 상쾌한 선택  
### **Brewer-Clifton 2012 3-D Sparkling Chardonnay**
**90점 · $68**

레몬 제스트, 라임, 레몬 속껍질의 강한 산미와 효모, 브리 치즈 껍질 같은 풍미가 있어 시저 샐러드의 상큼한 소스와 잘 맞습니다. 다만 리뷰상 입안이 매우 날카롭고 약간 시큼한 스타일이므로, 허브 시어링 스테이크보다는 **샐러드 비중이 큰 식사**에 더 적합합니다.

**추천 순서:**  
1. *

In [40]:
response = chain.invoke({
    'text': "이따 피자스쿨 베이컨포테이토 피자 먹을거다. 와인은 뭘 먹으면 좋을까?",
    "image_urls":[
       "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyMTExMjZfOTAg%2FMDAxNjM3ODc3MTI3MDY4.GrivVlwn1aHP8ydQZIA7mym_2AUzq7UaB8zqoTc1d8Mg.z6fe2yK3Tq9E95fHOpHJBq-c3fRUUTGPemiSY939QBcg.JPEG.kkuljo%2F20200406_210410.jpg&type=sc960_832"
    ]
})

print(response)

### 추천: **Tenuta Carretta 2007 Bric Quercia Barbera d’Alba**

베이컨포테이토 피자에는 제시된 와인 중 이 와인이 가장 잘 맞습니다.

- **치즈와의 조화:** 리뷰에 “매우 치즈가 많은 음식과 페어링하라”고 명시되어 있어, 피자의 치즈 풍미를 직접적으로 받쳐줍니다.
- **베이컨과의 연결:** 액체 연기, 구운 헤이즐넛, 숙성육의 향이 있어 베이컨의 짭짤하고 훈연된 풍미와 자연스럽게 이어집니다.
- **느끼함 보완:** 바르베라 특유의 높은 산도가 치즈와 베이컨의 기름기를 잘라내어, 산뜻한 샤르도네가 해주는 ‘말끔한 정리’에 가장 가까운 역할을 합니다.

다만 이 와인은 **알코올감이 높고 산도가 다소 강하다**는 점이 있어, 섬세하고 가벼운 피자보다는 치즈와 베이컨이 넉넉한 스타일에 더 적합합니다.

### 차선책: **Pecchenino 2011 Quass Barbera d’Alba**

검은 체리, 라즈베리, 후추, 허브와 로스팅 커피 향이 있어 베이컨의 구운 풍미와 잘 연결될 수 있습니다. 또한 탄탄한 산도가 있어 피자의 기름기를 보완합니다. 하지만 치즈와의 직접적인 궁합은 Bric Quercia보다 리뷰상 덜 분명합니다.

### 피하는 편이 좋은 선택

- **Deltetto 2012 Bussia Barolo:** “극도로 떫은 타닌”이 있어 짠 치즈와 피자의 부드러운 감촉을 거칠게 만들 수 있습니다.
- **Nada Giuseppe 2009 Casot Barbaresco:** 숙성된 베이컨과 어두운 과실 풍미는 흥미롭지만, 타닌의 먼지감과 익은 인상이 피자의 산뜻함을 살리기에는 다소 무겁습니다.

**최종 선택은 Bric Quercia입니다.** 산도가 치즈와 베이컨의 짠맛·기름기를 씻어내고, 훈연·숙성육 풍미가 베이컨과 맞물려 피자의 풍미를 확장합니다.


In [42]:
response = chain.invoke({
    'text': "와인 폭탄주",
})

print(response)

### 추천: Tornatore 2015 Bianco, Etna — Carricante

제시된 풍미가 **과실의 달콤함과 탄산의 강한 자극**, 그리고 **알코올의 짜릿하고 달콤한 여운**을 중심으로 한다면, 가장 잘 맞는 선택은 **Tornatore Etna Bianco**입니다.

- **신선한 산도와 시트러스**가 과실의 단맛과 알코올감을 산뜻하게 정리합니다.
- **그린 애플과 미네랄 풍미**가 탄산의 톡 쏘는 느낌과 자연스럽게 이어져, 맛을 더 선명하게 만듭니다.
- 와인이 **크리스프하고 직선적**이어서, 달콤하고 자극적인 풍미를 덮기보다 입안을 깨끗하게 씻어 다음 한 모금을 준비해 줍니다.
- 플린티한 미네랄 여운은 과실 향만 강조하는 대신 전체 인상을 보다 날렵하고 정돈되게 만들어 줍니다.

### 차선: Villa Raiano 2013 Alimata Fiano di Avellino

조금 더 풍성하고 깊이 있는 조화를 원한다면 **Fiano di Avellino**도 좋습니다. 강한 산도와 시트러스가 단맛을 보완하면서, 구조감과 허브·미네랄 풍미가 복합성을 더합니다. 다만 제시된 풍미처럼 탄산감이 특히 강조된다면, 더 가볍고 선명한 **Etna Bianco**가 우선입니다.

반면 **Agnitio Pinot Noir**는 과숙한 자두와 오크, 묵직한 바디가 있어 달콤하고 탄산감 있는 풍미를 무겁게 만들 수 있으며, **Au Bon Climat Pinot Noir**는 산도는 뛰어나지만 제시된 조합에는 레드 와인의 과실·콜라 풍미가 다소 과할 수 있습니다.


In [44]:
response = chain.invoke({
    'text': "고추 잡채",
    'image_url' : [
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMDNfNDYg%2FMDAxNzU5NDkwODcxMjE4.nDH5zOdTi5HvhLUtgYXtl9ps1jQLWn5MNC1XNrChF8Yg.xADP7ofugONTY3vwbnFeS7cQQmPTBXQfjC87WNsFF3Ig.JPEG%2FIMG_3801.JPG&type=sc960_832"
    ]
})

print(response)

## 추천: **Cass 2014 Mourvèdre (Paso Robles)**

고추잡채에는 **Cass 2014 Mourvèdre**가 가장 잘 어울립니다.

- **부드러운 고기의 감칠맛** ↔ 리뷰에 언급된 *meaty funk*와 *marjoram-crusted roast beef* 풍미가 고기의 깊은 맛을 자연스럽게 이어줍니다.
- **은은한 불향** ↔ 짙은 검은 과실과 허브 뉘앙스가 볶음 요리의 구수하고 그을린 풍미를 받쳐줍니다.
- **아삭한 피망** ↔ 충분한 산도가 피망의 산뜻함을 살리고, 기름기와 고기의 무게감을 깔끔하게 정리합니다.
- 리뷰상 와인이 **약간 달게 느껴진다**는 점도 장점입니다. 고추잡채의 짭짤하고 감칠맛 나는 풍미를 둥글게 감싸며, 타닌의 거친 인상을 완화해줄 수 있습니다.

다만 이 와인은 검은 과실과 산도가 비교적 뚜렷하므로, 고추잡채를 너무 맵거나 달게 만들기보다는 **담백하고 불향 중심으로 조리했을 때** 가장 좋습니다. 서빙 전 잠시 디캔팅하면 고기 풍미와 허브 향이 더 부드럽게 어우러집니다.

### 다른 후보와 비교
- **Wild Coyote Syrah**: 강한 후추 향과 무거운 타닌이 피망의 풋향과 충돌하거나 요리를 압도할 수 있습니다.
- **Wrath Syrah**: 리뷰에서도 소시지처럼 짠 고기를 필요로 한다고 설명되어, 현재의 고추잡채에는 다소 과합니다.
- **Aglianico**: 쓴맛·산도·타닌이 모두 강해 스테이크와 블루치즈에 더 적합합니다.
- **Pagos de Valcerracin Tempranillo**: 산미와 절인 양배추 같은 결점 풍미가 고추잡채의 섬세한 불향을 해칠 가능성이 큽니다.
